## Teacher Trajectory Collection

This is the realistic distillation path for our first blog.

We do **not** train on the held-out eval tasks. We also do **not** create training rows directly from BFCL's ground-truth tool calls. That simulator-only route can be useful for debugging, but it is not teacher distillation, so it is intentionally out of this notebook now.

Instead, we ask a stronger teacher model to run only on the train tasks. The teacher uses the exact same harness as the student: same prompts, same tool parser, same simulator, same BFCL scorer. If a teacher attempt solves the task, we keep that successful trajectory as supervised training data for the small model.

The data flow is:

```text
train task -> harness prompt -> teacher model -> parsed tool calls -> simulator -> BFCL score
successful teacher trajectory -> SFT rows for the student
held-out eval tasks -> untouched until final evaluation
```

For the teacher runtime, we use a raw MLX server around `mlx-community/Qwen3.5-35B-A3B-4bit`. This keeps us on the Qwen3.5 tokenizer family for the whole series, keeps MLX speed, and avoids the fragile OpenAI-style tool-call extraction layer. Start the server in a separate terminal before running the collection cell:

```bash
uv run python scripts/serve_teacher_mlx_raw.py
```

One important debugging detail: the official MLX-LM OpenAI-compatible endpoints can internally detect Qwen tool calls and return an empty text/message even though `finish_reason="tool_calls"`. The raw server avoids that layer. It returns the generated Qwen text directly, and the shared parser below extracts `<tool_call>` blocks.


This split notebook runs only the teacher-on-train trajectory collection part. Teacher eval is in `02_teacher_eval.ipynb`, and fine-tuning will live in later notebooks.


In [ ]:
from pathlib import Path
import json
import sys

cwd = Path.cwd()
if (cwd / "utils.py").exists():
    blog_dir = cwd
elif (cwd / "1-distilling-a-0-8b-tool-calling-agent" / "utils.py").exists():
    blog_dir = cwd / "1-distilling-a-0-8b-tool-calling-agent"
else:
    raise RuntimeError("Run this notebook from the repo root or the blog folder.")

if str(blog_dir) not in sys.path:
    sys.path.insert(0, str(blog_dir))

import utils as u

print("Blog dir:", blog_dir)
print("Output dir:", u.OUTPUT_DIR)
print("Student model:", u.STUDENT_MODEL)
print("Teacher model:", u.TEACHER_MODEL)


In [ ]:
split = u.load_bfcl_split()

print("Full BFCL v4 multi-turn base size:", len(split.all_entries))
print("Train split:", len(split.train_entries))
print("Held-out eval split:", len(split.eval_entries))
print("Current eval run:", len(split.benchmark_entries))
print("First train id:", split.train_entries[0]["id"])
print("First eval id:", split.benchmark_entries[0]["id"])
print("MAX_STEPS_PER_TURN:", u.MAX_STEPS_PER_TURN)
print("MAX_CONSECUTIVE_EXECUTION_ERRORS:", u.MAX_CONSECUTIVE_EXECUTION_ERRORS)


## From One Solved Task To Many Training Rows

A BFCL task can have multiple user turns. Inside each user turn, the teacher may need multiple tool calls before it can stop and wait for the next user request.

For sequence-level distillation, we keep only teacher trajectories that solve the **whole task**. Then we slice the successful trajectory into SFT rows. Each row trains the student on one next assistant action:

```text
input: conversation state before a teacher action
label: that teacher action
```

![Teacher trajectory converted into SFT rows](assets/teacher-trajectory-to-sft-rows.svg)

So yes: if a second user turn happens after the first user turn is complete, the training rows inside the second user turn include the full previous conversation history. That means the prompt for a second-turn action contains user turn 1, teacher tool calls from turn 1, tool observations from turn 1, the final answer for turn 1 if the teacher produced one, then user turn 2 and any tool observations already produced in turn 2.


## Inspect The Multi-Turn Shape

The next cell prints the first train task at the level we care about for training data: user turns and expected tool calls. This is not the data we train on directly, but it shows why one benchmark task can produce many SFT rows.

In [ ]:
example = split.train_entries[0]
answer = split.train_answers[0]

print("Task id:", example["id"])
print("User turns:", len(example["question"]))
print("Ground-truth tool calls per user turn:", [len(turn) for turn in answer["ground_truth"]])
print("Total ground-truth tool calls:", sum(len(turn) for turn in answer["ground_truth"]))
print()

for turn_index, (user_messages, expected_calls) in enumerate(
    zip(example["question"], answer["ground_truth"]),
    start=1,
):
    print(f"User turn {turn_index}")
    for message in user_messages:
        print("  user:", message["content"])
    print("  expected tool calls:")
    for call in expected_calls:
        print("   -", call)
    print()


## Configure Teacher Collection

Start the MLX teacher server in another terminal before running the collection cell:

```bash
uv run python scripts/serve_teacher_mlx_raw.py
```

Collection keeps the adaptive retry strategy because this notebook is not reporting a held-out eval metric. It is doing teacher-side rejection sampling to build training data: run a complete trajectory, score it with BFCL, keep the first valid trajectory, and skip the task if all attempts fail.

The eval notebooks should still use pass@1 as their main number. This notebook is where adaptive pass@5 belongs by default.

The three output files are also the resume cache. Once a train task has recorded attempts, rerunning this cell skips that task id. This includes failed tasks, which is intentional: if we change the retry strategy later, we should write to a new output filename rather than silently mixing strategies.


In [ ]:
tokenizer = u.load_tokenizer()
teacher_config = u.TeacherConfig()
LOG_TRACES_TO_MLFLOW = True
mlflow_config = u.MlflowConfig(enabled=LOG_TRACES_TO_MLFLOW)
server_health = u.teacher_server_health(teacher_config)
active_teacher_model = (server_health or {}).get("model", teacher_config.model_name)
TEACHER_RUN_LIMIT = 150
teacher_attempt_preview = u.adaptive_temperature_retry_attempts_for_task(0)

print("Teacher provider:", teacher_config.provider)
print("Teacher server:", teacher_config.server_base_url)
print("Teacher request model:", teacher_config.request_model)
print("Active teacher model:", active_teacher_model)
print("Train tasks to attempt:", TEACHER_RUN_LIMIT)
print("Retry attempts per task:", len(teacher_attempt_preview))
print("Retry schedule:")
print(json.dumps([u.attempt_config_to_dict(attempt) for attempt in teacher_attempt_preview], indent=2))
print("Teacher runtime ready:", u.teacher_runtime_is_configured(teacher_config))
print("MLflow logging enabled:", mlflow_config.enabled)
print("MLflow tracking URI:", mlflow_config.tracking_uri)
print("MLflow experiment:", mlflow_config.experiment_name)
print()
print("Cached local model candidates:")
for model_id in u.list_cached_huggingface_models():
    print(" -", model_id)
print()
print("To switch teacher model, stop the server and restart it, for example:")
print("uv run python scripts/serve_teacher_mlx_raw.py --model mlx-community/Qwen3.5-9B-MLX-8bit --port 8080")


## Collect Successful Teacher Trajectories

This is the expensive cell. It writes three artifacts:

- attempt summaries, including compact traces for failures;
- full successful traces;
- JSONL SFT rows built from successful teacher tool actions.


In [ ]:
if not u.teacher_runtime_is_configured(teacher_config):
    print("Teacher server is not ready. Start it with: uv run python scripts/serve_teacher_mlx_raw.py")
else:
    teacher_slug = u.model_slug(active_teacher_model)
    teacher_attempts_path = u.OUTPUT_DIR / f"{teacher_slug}_bfcl_v4_multi_turn_base_train_attempts_raw_qwen_parser_v2_adaptive_temp_ramp.json"
    teacher_traces_path = u.OUTPUT_DIR / f"{teacher_slug}_bfcl_v4_multi_turn_base_successful_traces_raw_qwen_parser_v2_adaptive_temp_ramp.json"
    teacher_sft_rows_path = u.OUTPUT_DIR / f"{teacher_slug}_bfcl_v4_multi_turn_base_successful_sft_rows_raw_qwen_parser_v2_adaptive_temp_ramp.jsonl"

    generate_teacher_action_factory = u.make_teacher_action_generator_factory(teacher_config)
    teacher_collection = u.collect_successful_teacher_trajectories(
        split.train_entries,
        split.train_answers,
        tokenizer=tokenizer,
        run_limit=TEACHER_RUN_LIMIT,
        generate_action_factory=generate_teacher_action_factory,
        attempt_configs_factory=u.adaptive_temperature_retry_attempts_for_task,
        attempts_output_path=teacher_attempts_path,
        traces_output_path=teacher_traces_path,
        sft_rows_output_path=teacher_sft_rows_path,
        mlflow_config=mlflow_config,
        mlflow_tags={"model_role": "teacher", "model_name": active_teacher_model, "provider": teacher_config.provider, "run_type": "train_trajectory_collection"},
    )

    successful_task_count = len(teacher_collection["selected_traces"])
    attempted_task_count = min(TEACHER_RUN_LIMIT, len(split.train_entries))
    completed_task_count = len({row["id"] for row in teacher_collection["attempts"]})
    sft_row_count = len(teacher_collection["sft_rows"])

    print("\nTeacher requested train tasks:", attempted_task_count)
    print("Teacher completed train tasks:", completed_task_count)
    print("Successful teacher trajectories:", successful_task_count)
    print("Teacher SFT rows:", sft_row_count)
    print("Saved teacher attempts to:", teacher_attempts_path)
    print("Saved successful teacher traces to:", teacher_traces_path)
    print("Saved teacher SFT rows to:", teacher_sft_rows_path)


What we want to see after running this is not just "the teacher is better." We want a dataset with the exact action traces the student should imitate.

If the teacher only succeeds on some train tasks, that is still useful. We keep the successful traces, inspect the failures, and then decide whether to add a real search strategy: varied seeds, higher temperature, prompt variants, or a stronger teacher backend. The held-out 50 eval tasks remain untouched until after fine-tuning.
